In [25]:
import os
import json
import time
import torch
import math
import glob
import threading
import queue
import collections
from dataclasses import dataclass
from typing import Deque, Dict, List, Optional, Tuple


import cv2
import numpy as np

In [26]:
# ------------------------------
# 性能优化：尽可能释放硬件性能
# ------------------------------
try:
    import torch
except Exception:
    torch = None


# 尝试 DirectML（Windows + AMD/NVIDIA/Intel 通用加速）
try:
    import torch_directml as _tdml # 若不可用会抛异常
    _HAS_DML = True
except Exception:
    _tdml = None
    _HAS_DML = False

In [27]:
# ------------------------------
# 配置区（可直接修改；也支持环境变量覆盖）
# ------------------------------
CONFIG = {
    # --- 路径 ---
    # 可填具体文件；也可填一个目录，脚本会在其中挑选**最新**的 mp4/mov/mkv
    "VIDEO_PATH": os.getenv("VIDEO_TEST_PATH", "../vid/yuwenbin.mp4"),
    "CKPT_PATH": os.getenv("CKPT_PATH", "../src/runs/hseq/best_state.pth"),
    # 可为空：若为空则自动写到与视频同目录下，命名为 <视频名>_annotated.mp4
    "OUT_PATH": os.getenv("OUT_PATH", ""),
    # 可选：标签映射（支持 {id:name} 或 {name:id}）
    "LABEL_MAP_PATH": os.getenv("LABEL_MAP", ""),
    
    
    # --- 模型/特征 ---
    # 'torchscript' 或 'python'
    "MODEL_TYPE": os.getenv("MODEL_TYPE", "torchscript"),
    # 当 MODEL_TYPE='python' 时，需要提供 Python 类：'pkg.module:Class'
    "MODEL_CLASS": os.getenv("MODEL_CLASS", ""),
    # JSON 字符串或直接在此写 dict：例如 {"feature_dim":128, "num_classes":7}
    "MODEL_ARGS": json.loads(os.getenv("MODEL_ARGS", "{}")),
    
    
    # 特征流水线：不填则使用内置默认（32 维，能跑通）
    "FEATURE_CLASS": os.getenv("FEATURE_CLASS", ""), # 'pkg.module:FeatureClass'
    "FEATURE_ARGS": json.loads(os.getenv("FEATURE_ARGS", "{}")),
    
    
    # --- 时序/渲染 ---
    "NUM_CLASSES": int(os.getenv("NUM_CLASSES", "7")),
    "FEATURE_DIM": int(os.getenv("FEATURE_DIM", "32")),
    "TARGET_FPS": float(os.getenv("TARGET_FPS", "10")), # 推理帧率（源视频将按此下采样）
    "WINDOW_SEC": float(os.getenv("WINDOW_SEC", "2.0")), # 滑动窗口秒数（与训练一致）
    "STRIDE_SEC": float(os.getenv("STRIDE_SEC", "0.2")), # 每次推理间隔秒数
    "SMOOTHING": float(os.getenv("SMOOTHING", "0.6")), # 概率 EMA 平滑系数 [0..1]
    
    
    # --- 设备 ---
    # 'auto' | 'cuda' | 'dml' | 'cpu' （ROCm 下 torch.cuda.is_available==True）
    "DEVICE": os.getenv("DEVICE", "auto"),
    # 是否尝试 torch.compile 加速（需要 PyTorch>=2.0）
    "USE_COMPILE": os.getenv("USE_COMPILE", "1") == "1",
    # AMP 精度：'fp16' | 'bf16' | ''
    "AMP": os.getenv("AMP", ""),
    
    
    # --- IO/解码 ---
    "PREFETCH_FRAMES": int(os.getenv("PREFETCH_FRAMES", "96")), # 读帧预取队列
    "WRITE_ENABLED": os.getenv("WRITE_ENABLED", "1") == "1", # 默认开启写文件
}

In [28]:
# ------------------------------
# 实用函数：路径解析 / 动态导入 / JSON 解析
# ------------------------------
import importlib


def resolve_class(spec: str):
    """按 'pkg.module:Class' 载入类对象。"""
    if not spec or ':' not in spec:
        raise ValueError("类路径需为 'pkg.module:Class' 格式")
    mod, cls = spec.split(':', 1)
    m = importlib.import_module(mod)
    return getattr(m, cls)




def resolve_video_path(path_or_dir: str) -> str:
    """允许传目录：自动挑最新的视频文件；也支持具体文件路径。"""
    p = os.path.expanduser(path_or_dir)
    if os.path.isdir(p):
        cand = []
        for ext in ("*.mp4", "*.mov", "*.mkv", "*.avi"):
            cand.extend(glob.glob(os.path.join(p, ext)))
        if not cand:
            raise FileNotFoundError(f"目录下未找到视频：{p}")
        cand.sort(key=lambda x: os.path.getmtime(x), reverse=True)
        return cand[0]
    if not os.path.exists(p):
        raise FileNotFoundError(f"未找到视频：{p}")
    return p




def default_out_path(video_path: str) -> str:
    d = os.path.dirname(video_path)
    base = os.path.splitext(os.path.basename(video_path))[0]
    return os.path.join(d, f"{base}_annotated.mp4")

In [29]:
# ------------------------------
# 标签映射
# ------------------------------


def load_label_map(path: Optional[str]) -> Dict[int, str]:
    default = {0: "clean", 1: "yaw_large", 2: "pitch_down", 3: "face_not_visible",
        4: "other_person", 5: "suspicious_body", 6: "leave_seat"}
    if not path:
        return default
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    # 兼容 {id:name} 或 {name:id}
    if all(isinstance(k, str) for k in data.keys()):
        inv = {int(v): k for k, v in data.items()}
        return inv
    return {int(k): str(v) for k, v in data.items()}

In [30]:
# ------------------------------
# 默认特征流水线（演示用，可被自定义类替代）
# ------------------------------
class DefaultFeaturePipeline:
    """将 BGR 帧转为 32 维轻量特征（灰度直方图 + 梯度强度）。
    提示：为追求真实效果，请使用你训练时的一致特征，见 FEATURE_CLASS。"""
    def __init__(self, device: str = "cpu"):
        self.device = device
    
    
    def __call__(self, frame_bgr: np.ndarray) -> np.ndarray:
        g = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        g = cv2.resize(g, (64, 64), interpolation=cv2.INTER_AREA)
        hist = cv2.calcHist([g], [0], None, [31], [0, 256]).flatten()
        hist = hist / (hist.sum() + 1e-6)
        sx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
        sy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
        motion = float(np.mean(np.hypot(sx, sy)))
        feat = np.concatenate([hist.astype(np.float32), np.array([motion], dtype=np.float32)])
        # 填充到 32 维
        if feat.shape[0] < 32:
            feat = np.pad(feat, (0, 32 - feat.shape[0]))
        elif feat.shape[0] > 32:
            feat = feat[:32]
        return feat.astype(np.float32)

In [31]:
# ------------------------------
# 模型执行器（支持 TorchScript / Python 类 + state_dict）
# ------------------------------
class ModelRunner:
    """滑动窗口推理执行器。
    - 支持两种加载方式：
      1) TorchScript：MODEL_TYPE='torchscript'，直接 `torch.jit.load(CKPT_PATH)`。
      2) Python 类：MODEL_TYPE='python'，`MODEL_CLASS='pkg.mod:Class'` + `MODEL_ARGS`，
         然后从 `CKPT_PATH` 里加载 state_dict（兼容常见前缀）。
    - 输入：[T, F]，输出：[C] 概率（若模型输出 [1, T, C]，取最后时刻）。
    """

    def __init__(self, ckpt_path: str, feature_dim: int, window: int, device: str,
                 num_classes: int, model_type: str,
                 model_class: Optional[str] = None, model_args: Optional[dict] = None,
                 use_compile: bool = True, amp: str = ""):
        self.ckpt_path = ckpt_path
        self.feature_dim = feature_dim
        self.window = window
        self.num_classes = int(num_classes)
        self.buf: Deque[np.ndarray] = collections.deque(maxlen=window)
        self.model = None
        self.device_kind = device  # 'cuda' | 'dml' | 'cpu'
        self.amp = amp.lower()

        # 设备实例
        self._dml_device = None
        if torch is None:
            print("[警告] 未安装 PyTorch，转为伪概率输出（仅演示 HUD）。")
        else:
            if self.device_kind == 'dml':
                if not _HAS_DML:
                    raise RuntimeError("选择了 dml 设备，但未安装 torch-directml。")
                self._dml_device = _tdml.device()

            # 加载模型
            if model_type == 'torchscript':
                self.model = torch.jit.load(ckpt_path, map_location='cpu')
            elif model_type == 'python':
                if not model_class:
                    raise ValueError("MODEL_TYPE='python' 需要提供 MODEL_CLASS='pkg.mod:Class'")
                Klass = resolve_class(model_class)
                kwargs = dict(model_args or {})
                kwargs.setdefault("feature_dim", feature_dim)
                kwargs.setdefault("num_classes", self.num_classes)
                self.model = Klass(**kwargs)
                state = torch.load(ckpt_path, map_location='cpu')
                sd = state.get("state_dict", state)
                # 兼容前缀
                new_sd = {}
                for k, v in sd.items():
                    nk = k
                    for pref in ("model.", "module.", "net."):
                        if nk.startswith(pref):
                            nk = nk[len(pref):]
                    new_sd[nk] = v
                missing, unexpected = self.model.load_state_dict(new_sd, strict=False)
                if missing:   print("[load_state_dict] 缺失键：", missing)
                if unexpected:print("[load_state_dict] 多余键：", unexpected)
            else:
                raise ValueError("MODEL_TYPE 仅支持 'torchscript' 或 'python'")

            self.model.eval()

            # 放到目标设备
            if self.device_kind == 'cuda' and torch.cuda.is_available():
                self.model.to('cuda')
                # cuDNN benchmark：固定分辨率时更快
                try:
                    import torch.backends.cudnn as cudnn
                    if cudnn.is_available():
                        cudnn.benchmark = True
                except Exception:
                    pass
            elif self.device_kind == 'dml':
                self.model.to(self._dml_device)
            else:
                self.device_kind = 'cpu'

            # 编译加速（PyTorch 2.x）
            if use_compile and hasattr(torch, 'compile'):
                try:
                    self.model = torch.compile(self.model, mode='reduce-overhead')
                    print("[compile] 已启用 torch.compile 加速")
                except Exception as e:
                    print("[compile] 跳过：", e)

    def push(self, feat: np.ndarray) -> None:
        assert feat.shape[-1] == self.feature_dim, (
            f"特征维度不符：got {feat.shape[-1]}, expect {self.feature_dim}")
        self.buf.append(feat)

    def ready(self) -> bool:
        return len(self.buf) == self.window

    def _to_device(self, x: 'torch.Tensor'):
        if self.device_kind == 'cuda':
            return x.cuda(non_blocking=True)
        if self.device_kind == 'dml':
            return x.to(self._dml_device)
        return x

    def _autocast_ctx(self):
        if torch is None:
            from contextlib import nullcontext
            return nullcontext()
        if self.device_kind == 'cuda' and self.amp in ('fp16', 'bf16'):
            dtype = torch.float16 if self.amp == 'fp16' else torch.bfloat16
            return torch.cuda.amp.autocast(dtype=dtype)
        # 其他设备默认不开 AMP
        from contextlib import nullcontext
        return nullcontext()

    @staticmethod
    def _softmax_np(logits: np.ndarray) -> np.ndarray:
        e = np.exp(logits - logits.max())
        return (e / e.sum()).astype(np.float32)

    def predict_proba(self) -> np.ndarray:
        if not self.ready():
            return np.zeros((self.num_classes,), dtype=np.float32)

        if self.model is None or torch is None:
            # 无模型/无 torch 的演示概率
            return self._softmax_np(np.zeros((self.num_classes,), dtype=np.float32))

        x = np.stack(list(self.buf), axis=0)  # [T, F]
        xt = torch.from_numpy(x).float().unsqueeze(0)  # [1, T, F]
        xt = self._to_device(xt)

        with torch.inference_mode():
            with self._autocast_ctx():
                out = self.model(xt)
        if isinstance(out, (list, tuple)):
            out = out[-1]
        if out.ndim == 3:
            out = out[:, -1]
        prob = torch.softmax(out, dim=-1).squeeze(0).detach().cpu().numpy().astype(np.float32)
        if not np.all(np.isfinite(prob)):
            prob = np.nan_to_num(prob, nan=0.0, posinf=0.0, neginf=0.0)
            s = prob.sum()
            prob = prob / s if s > 0 else np.full_like(prob, 1.0 / len(prob))
        return prob



In [32]:
# ------------------------------
# 事件聚合（双阈值滞回 + 最小时长 + 优先级）
# ------------------------------
@dataclass
class ClassRule:
    start_th: float
    stop_th: float
    min_dur_sec: float
    priority: int  # 越大优先级越高


class EventAggregator:
    def __init__(self, label_map: Dict[int, str], fps: float, rules: Dict[int, ClassRule]):
        self.label_map = label_map
        self.fps = fps
        self.rules = rules
        self.active_cls: Optional[int] = None
        self.active_start_f: Optional[int] = None
        self.frame_idx = 0
        self.events: List[Dict] = []

    def update(self, probs: np.ndarray) -> Tuple[Optional[int], Optional[Dict]]:
        self.frame_idx += 1
        candidates = []
        for cid, p in enumerate(probs):
            rule = self.rules.get(cid)
            if rule is None:
                continue
            if self.active_cls is None:
                if p >= rule.start_th:
                    candidates.append((rule.priority, cid))
            else:
                if cid == self.active_cls and p >= rule.stop_th:
                    candidates.append((rule.priority, cid))
        new_active = None
        if candidates:
            candidates.sort(reverse=True)
            new_active = candidates[0][1]

        ended_event = None
        if self.active_cls is None and new_active is not None:
            self.active_cls = new_active
            self.active_start_f = self.frame_idx
        elif self.active_cls is not None:
            if new_active == self.active_cls:
                pass
            else:
                start_f = self.active_start_f or (self.frame_idx - 1)
                dur_sec = (self.frame_idx - start_f) / max(self.fps, 1.0)
                cid = self.active_cls
                rule = self.rules.get(cid)
                if rule and dur_sec >= rule.min_dur_sec:
                    ended_event = {
                        "class_id": cid,
                        "class_name": self.label_map.get(cid, str(cid)),
                        "start_frame": int(start_f),
                        "end_frame": int(self.frame_idx - 1),
                        "duration_sec": float(dur_sec),
                    }
                    self.events.append(ended_event)
                if new_active is not None:
                    self.active_cls = new_active
                    self.active_start_f = self.frame_idx
                else:
                    self.active_cls = None
                    self.active_start_f = None
        return self.active_cls, ended_event

    def flush(self) -> Optional[Dict]:
        if self.active_cls is None or self.active_start_f is None:
            return None
        dur_sec = (self.frame_idx - self.active_start_f + 1) / max(self.fps, 1.0)
        cid = self.active_cls
        rule = self.rules.get(cid)
        if rule and dur_sec >= rule.min_dur_sec:
            ev = {
                "class_id": cid,
                "class_name": self.label_map.get(cid, str(cid)),
                "start_frame": int(self.active_start_f),
                "end_frame": int(self.frame_idx),
                "duration_sec": float(dur_sec),
            }
            self.events.append(ev)
            self.active_cls = None
            self.active_start_f = None
            return ev
        return None



In [33]:
# ------------------------------
# HUD 绘制（叠加到视频帧上）
# ------------------------------
@dataclass
class HudStyle:
    font: int = cv2.FONT_HERSHEY_SIMPLEX
    fscale: float = 0.6
    thick: int = 2
    pad: int = 8
    line: int = 18


def draw_hud(frame: np.ndarray, probs: np.ndarray, label_map: Dict[int, str],
             active_cls: Optional[int], style: HudStyle, fps_meas: float,
             timeline: Deque[int], class_colors: Dict[int, Tuple[int, int, int]]):
    h, w = frame.shape[:2]
    x, y = style.pad, style.pad + 12
    cv2.putText(frame, f"Realtime Validator | FPS: {fps_meas:.1f}", (x, y),
                style.font, style.fscale, (255, 255, 255), style.thick, cv2.LINE_AA)
    y += style.line
    if probs is not None and probs.size > 0:
        tops = np.argsort(probs)[-3:][::-1]
        for rank, cid in enumerate(tops, 1):
            p = float(probs[cid])
            name = label_map.get(int(cid), str(cid))
            color = class_colors.get(int(cid), (200, 200, 200))
            cv2.putText(frame, f"{rank}. {name}: {p:.2f}", (x, y),
                        style.font, style.fscale, color, style.thick, cv2.LINE_AA)
            y += style.line
    if active_cls is not None:
        name = label_map.get(int(active_cls), str(active_cls))
        color = class_colors.get(int(active_cls), (0, 200, 255))
        cv2.rectangle(frame, (0, h - 40), (w, h), color, thickness=-1)
        cv2.putText(frame, f"ACTIVE: {name}", (style.pad, h - 12),
                    style.font, 0.7, (0, 0, 0), 2, cv2.LINE_AA)
    if timeline:
        bar_h = 8
        bar_w = min(len(timeline), w)
        start_x = w - bar_w
        y0 = h - 48
        for i, cid in enumerate(list(timeline)[-bar_w:]):
            c = class_colors.get(int(cid), (120, 120, 120))
            cv2.line(frame, (start_x + i, y0), (start_x + i, y0 + bar_h), c, 1)
        cv2.rectangle(frame, (start_x - 1, y0 - 1), (w, y0 + bar_h + 1), (220, 220, 220), 1)



In [34]:
# ------------------------------
# 事件导出
# ------------------------------

def save_events_csv_json(events: List[Dict], out_prefix: str):
    import csv
    csv_path = out_prefix + "_events.csv"
    json_path = out_prefix + "_events.json"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["class_id", "class_name", "start_frame", "end_frame", "duration_sec"])
        for e in events:
            w.writerow([e["class_id"], e["class_name"], e["start_frame"], e["end_frame"], f"{e['duration_sec']:.3f}"])
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(events, f, ensure_ascii=False, indent=2)
    print(f"[导出] {csv_path}[导出] {json_path}")



In [35]:
# ------------------------------
# 高性能读帧（独立线程预取）
# ------------------------------
class FrameReader:
    """使用单独线程预取视频帧，减少主线程等待。"""
    def __init__(self, video_path: str, prefetch: int = 96):
        self.cap = cv2.VideoCapture(video_path)
        if not self.cap.isOpened():
            raise FileNotFoundError(f"无法打开视频：{video_path}")
        self.q: "queue.Queue[Optional[np.ndarray]]" = queue.Queue(maxsize=prefetch)
        self._stop = threading.Event()
        self.t = threading.Thread(target=self._loop, daemon=True)
        self.t.start()

    def _loop(self):
        while not self._stop.is_set():
            ok, frame = self.cap.read()
            if not ok:
                self.q.put(None)  # 结束标记
                break
            try:
                self.q.put(frame, timeout=0.5)
            except queue.Full:
                pass  # 队列满时丢弃，主线程会以最后帧为准

    def read(self) -> Tuple[bool, Optional[np.ndarray]]:
        try:
            item = self.q.get(timeout=1.0)
        except queue.Empty:
            return False, None
        if item is None:
            return False, None
        return True, item

    def release(self):
        self._stop.set()
        try:
            self.t.join(timeout=1.0)
        except Exception:
            pass
        self.cap.release()



In [36]:
# ------------------------------
# 主流程（无需 argparse）
# ------------------------------

def main():
    # 选择视频与输出路径
    video_path = resolve_video_path(CONFIG["VIDEO_PATH"])
    out_path = CONFIG["OUT_PATH"] or default_out_path(video_path)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    print(f"[输入视频] {video_path}")
    print(f"[导出视频] {out_path}")

    # 设备选择：尽量满血
    device = CONFIG["DEVICE"].lower()
    if device == 'auto':
        if torch is not None and getattr(torch, 'cuda', None) and torch.cuda.is_available():
            device = 'cuda'
        elif _HAS_DML:
            device = 'dml'
        else:
            device = 'cpu'
    print(f"[设备] {device}")

    # 线程/数学设置（CPU 路径优化）——避免在函数内再次 import torch 导致作用域冲突
    if 'torch' in globals() and torch is not None:
        try:
            if hasattr(torch, 'set_num_threads'):
                torch.set_num_threads(os.cpu_count() or 8)
            if hasattr(torch, 'set_float32_matmul_precision'):
                torch.set_float32_matmul_precision('high')
        except Exception:
            pass

    # 打开视频（线程预取）
    fr = FrameReader(video_path, prefetch=CONFIG["PREFETCH_FRAMES"])

    # 获取基础信息
    cap = fr.cap
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # 采样参数
    sample_every = max(1, int(round(src_fps / CONFIG["TARGET_FPS"])))
    eff_fps = src_fps / sample_every
    window = max(1, int(round(CONFIG["WINDOW_SEC"] * eff_fps)))
    stride_frames = max(1, int(round(CONFIG["STRIDE_SEC"] * eff_fps)))

    # 标签与颜色
    label_map = load_label_map(CONFIG["LABEL_MAP_PATH"]) if CONFIG["LABEL_MAP_PATH"] else load_label_map(None)
    rng = np.random.default_rng(42)
    class_colors = {cid: tuple(int(x) for x in rng.integers(60, 255, size=3)) for cid in label_map.keys()}

    # 规则（可按需调整）
    rules = {
        1: ClassRule(0.60, 0.45, 0.30, 1),
        2: ClassRule(0.60, 0.45, 0.30, 1),
        3: ClassRule(0.70, 0.55, 0.50, 2),
        4: ClassRule(0.70, 0.55, 0.30, 2),
        5: ClassRule(0.70, 0.55, 0.30, 2),
        6: ClassRule(0.70, 0.55, 3.50, 3),
    }

    # 特征管线（自定义 > 默认）
    if CONFIG["FEATURE_CLASS"]:
        FeatureCls = resolve_class(CONFIG["FEATURE_CLASS"])
        feat_pipe = FeatureCls(**CONFIG["FEATURE_ARGS"])  # 你的训练时特征
        feature_dim = int(CONFIG["FEATURE_DIM"])  # 与训练一致
    else:
        feat_pipe = DefaultFeaturePipeline(device=device)
        feature_dim = int(CONFIG["FEATURE_DIM"])  # 默认 32

    # 模型执行器
    runner = ModelRunner(
        ckpt_path=CONFIG["CKPT_PATH"],
        feature_dim=feature_dim,
        window=window,
        device=device,
        num_classes=int(CONFIG["NUM_CLASSES"]),
        model_type=CONFIG["MODEL_TYPE"],
        model_class=(CONFIG["MODEL_CLASS"] or None),
        model_args=CONFIG["MODEL_ARGS"],
        use_compile=CONFIG["USE_COMPILE"],
        amp=CONFIG["AMP"],
    )

    # 写文件
    writer = None
    write_on = bool(CONFIG["WRITE_ENABLED"]) and bool(out_path)
    if write_on:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")  # 如需 H.264，可尝试 'avc1'（需本地编码器）
        writer = cv2.VideoWriter(out_path, fourcc, src_fps, (w, h))
        if not writer.isOpened():
            print("[警告] 无法打开视频写入器，自动关闭写文件")
            writer = None
            write_on = False

    # 状态变量
    ema = None
    alpha = float(CONFIG["SMOOTHING"])  # EMA 系数
    agg = EventAggregator(label_map, fps=eff_fps, rules=rules)
    timeline: Deque[int] = collections.deque(maxlen=int(eff_fps * 6))
    style = HudStyle()

    frame_idx = -1
    last_infer_idx = -stride_frames
    t0 = time.time()
    shown_fps = 0.0
    paused = False

    print("[开始] 实时渲染中…  (q 退出 / 空格 暂停 / s 截图 / w 切换写出)")

    while True:
        if not paused:
            ok, frame = fr.read()
            if not ok:
                break
            frame_idx += 1

            # 显示 FPS（每 ~0.5s 更新）
            if frame_idx % max(1, int(src_fps // 2)) == 0:
                dt = time.time() - t0
                if dt > 0:
                    shown_fps = frame_idx / dt

            # 按 TARGET_FPS 下采样参与推理
            if frame_idx % sample_every == 0:
                feat = feat_pipe(frame)
                runner.push(feat)
                if runner.ready() and ((frame_idx - last_infer_idx) >= stride_frames):
                    last_infer_idx = frame_idx
                    probs = runner.predict_proba()
                    if ema is None:
                        ema = probs
                    else:
                        ema = alpha * probs + (1 - alpha) * ema
                    active_cls, _ = agg.update(ema)
                else:
                    active_cls = agg.active_cls
            else:
                active_cls = agg.active_cls
                probs = ema if ema is not None else np.zeros((int(CONFIG["NUM_CLASSES"]),), dtype=np.float32)

            # 时间轴与 HUD
            timeline.append(active_cls if active_cls is not None else 0)
            draw_hud(frame, probs=ema if ema is not None else np.zeros((int(CONFIG["NUM_CLASSES"]),), dtype=np.float32),
                     label_map=label_map, active_cls=active_cls, style=style,
                     fps_meas=shown_fps, timeline=timeline, class_colors=class_colors)

            cv2.imshow("Realtime Validator", frame)
            if write_on and writer is not None:
                writer.write(frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord(' '):
            paused = not paused
        elif key == ord('s'):
            ts = int(time.time())
            out_img = os.path.join(os.path.dirname(out_path), f"hud_{ts}.png")
            cv2.imwrite(out_img, frame)
            print(f"[截图] {out_img}")
        elif key == ord('w'):
            write_on = not write_on
            print(f"[写出开关] {write_on}")

    # 结束清理
    flush_ev = agg.flush()
    if flush_ev:
        print("[flush]", flush_ev)

    out_prefix = os.path.splitext(out_path)[0] if out_path else os.path.splitext(os.path.basename(video_path))[0]
    save_events_csv_json(agg.events, out_prefix)

    fr.release()
    if writer is not None:
        writer.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()


[输入视频] ../vid/yuwenbin.mp4
[导出视频] ../vid\yuwenbin_annotated.mp4
[设备] cpu


RuntimeError: PytorchStreamReader failed locating file constants.pkl: file not found